<a href="https://colab.research.google.com/github/nguyenhoangnguyenazit-commits/ISYS2001-A2/blob/main/isys2012_a2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Budget Breaker
### ISYS2001 — Assessment 2: Programming Project

A finance assistant that helps university students in Perth track spending, understand where their money goes and check whether they're hitting their savings goal yet. Inspired by the "Spending break down" feature in the CommBank app.


## 1. Understand the Problem

**Business Problem:** As a university student living in Perth, it is easy to lose track of minor daily expenses like coffee, transport, and eating out. By the end of the month I often don't know where my money went and whether I hit my savings target.

**Solution:** A "Budget Breaker" finance assistant that automatically processes monthly transaction data, categorises expenses, calculates what percentage of income is spent per category, and reports whether the monthly savings goal was achieved. A Gemini-powered chat assistant then gives friendly, grounded advice based on those real numbers.

**Target User:** University students who need a quick, no-fuss way to track their living expenses and savings without manually calculating every receipt.

## 2. Identify the Inputs and Outputs

**Inputs:**
- `csv_source` (file path or file-like object): monthly transactions with columns `Date, Amount, Category, Description`
- `monthly_income` (float): total expected income for the month
- `saving_goal` (float): target amount to save by the end of the month

**Outputs:** a dictionary containing:
- `category_totals` — total spent per category
- `category_percentages` — % of income spent per category
- `total_spent` — sum of all expenses
- `remaining_balance` — `monthly_income - total_spent`
- `goal_achieved` — `True` if `remaining_balance >= saving_goal`


##3. Work an Example by Hand

## Step 3: Work an Example by Hand

Take a small sample of 4 transactions and work the maths manually, before writing any code.

| Date | Amount | Category |
|---|---|---|
| 2026-10-01 | 290.00 | Rent |
| 2026-10-01 | 5.20 | Coffee |
| 2026-10-02 | 42.30 | Groceries |
| 2026-10-09 | 6.10 | Coffee |

Assume `monthly_income = 1500`, `saving_goal = 300`.

**By hand:**
- Category totals: `Rent = 290.00`, `Coffee = 5.20 + 6.10 = 11.30`, `Groceries = 42.30`
- `total_spent = 290.00 + 11.30 + 42.30 = 343.60`
- `remaining_balance = 1500 - 343.60 = 1156.40`
- `goal_achieved = 1156.40 >= 300` → **True**
- Percentages: `Rent = 290.00/1500*100 = 19.33%`, `Coffee = 11.30/1500*100 = 0.75%`, `Groceries = 42.30/1500*100 = 2.82%`

This worked example is what Test 1 in the testing section checks against later, using a slightly larger sample.

##4. Pseudocode

```
FUNCTION break_down_budget(csv_source, monthly_income, saving_goal):
    TRY:
        READ csv_source into a table of rows (Date, Amount, Category, Description)
        IF required columns are missing THEN RAISE an error
        IF the table has no rows THEN RAISE an error
        CONVERT the Amount column to numbers
        IF any Amount fails to convert THEN RAISE an error
        IF monthly_income <= 0 THEN RAISE an error
        IF saving_goal < 0 THEN RAISE an error

        FOR each unique category in the table:
            category_totals[category] = SUM of Amount where Category = category

        total_spent = SUM of all values in category_totals
        remaining_balance = monthly_income - total_spent

        FOR each category, total in category_totals:
            category_percentages[category] = (total / monthly_income) * 100

        goal_achieved = (remaining_balance >= saving_goal)

        RETURN a dictionary with category_totals, category_percentages,
               total_spent, remaining_balance, goal_achieved

    CATCH any error:
        PRINT a clear error message
        RETURN None
```

##5. Convert to Python

Takes a CSV of transactions plus income and savings goal, returns a summary dictionary (category totals, percentages, remaining balance, goal status). Wrapped in try/except so bad input (missing file, bad columns, invalid numbers) returns None with a clear message instead of crashing.


In [ ]:
import pandas as pd
import io
import requests
import gradio as gr
import getpass

def break_down_budget(csv_source, monthly_income, saving_goal):
    """Analyse transactions and return a budget summary, or None if something is wrong."""
    try:
        df = pd.read_csv(csv_source)
    except Exception as e:
        print(f"Error processing budget: {e}")
        return None

    if "Amount" not in df.columns or "Category" not in df.columns:
        print("Error processing budget: CSV must have Amount and Category columns.")
        return None

    if df.empty:
        print("Error processing budget: CSV has no data.")
        return None

    amounts = list(df["Amount"])
    categories = list(df["Category"])

    # Check every amount is a real number (for loop + try/except, Module 2-5)
    valid = True
    for amt in amounts:
        try:
            float(amt)
        except ValueError:
            valid = False

    if not valid:
        print("Error processing budget: Amount column has invalid numbers.")
        return None

    if monthly_income <= 0:
        print("Error processing budget: Income must be greater than 0.")
        return None

    # Add up spending per category ourselves, using a dict + for loop (no groupby)
    category_totals = {}
    for i in range(len(categories)):
        cat = categories[i]
        amt = float(amounts[i])
        if cat in category_totals:
            category_totals[cat] = category_totals[cat] + amt
        else:
            category_totals[cat] = amt

    total_spent = sum(category_totals.values())
    remaining_balance = monthly_income - total_spent

    category_percentages = {}
    for cat, amt in category_totals.items():
        category_percentages[cat] = (amt / monthly_income) * 100

    goal_achieved = remaining_balance >= saving_goal

    return {
        "category_totals": category_totals,
        "category_percentages": category_percentages,
        "total_spent": total_spent,
        "remaining_balance": remaining_balance,
        "goal_achieved": goal_achieved
    }

## 6. Testing

Checks a normal case against hand-calculated numbers, plus one error case
(file that doesn't exist) to confirm the function fails safely.

In [ ]:
csv_data = """Date,Amount,Category
2023-10-01,290,Rent
2023-10-01,5.20,Coffee
2023-10-02,42.30,Groceries
"""

# Test 1: Happy path
result = break_down_budget(io.StringIO(csv_data), 1500, 300)
assert result["total_spent"] == 337.50
assert result["goal_achieved"] == True

# Test 2: File Inexistent
result_bad = break_down_budget("not_a_file.csv", 1500, 300)
assert result_bad is None

# Test 3: Missing Category
bad_csv = "Date,Amount\n2023-10-01,50\n"
result_missing = break_down_budget(io.StringIO(bad_csv), 1500, 300)
assert result_missing is None

# Test 4: Income = 0 or negative
result_bad_income = break_down_budget(io.StringIO(csv_data), 0, 300)
assert result_bad_income is None

print("Tests passed!")

Error processing budget: [Errno 2] No such file or directory: 'not_a_file.csv'
Error processing budget: CSV must have Amount and Category columns.
Error processing budget: Income must be greater than 0.
Tests passed!


## 7. Finance Assistant (Gemini)

Calls the Gemini API directly with `requests`. The persona/purpose is set
through a system instruction. When a budget summary is passed in, the
assistant's reply is grounded in the user's real data instead of giving
generic advice.

In [31]:
GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key: ")
GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent?key={GEMINI_API_KEY}"

PERSONA = "You are Budget Breaker, a friendly finance coach for students. Keep answers short and practical. If asked something off-topic, politely steer back to budgeting."

def ask_assistant(message, budget_summary=None):
    text = message
    if budget_summary:
        text += f"\n\nUser's budget data: {budget_summary}"
    payload = {
        "system_instruction": {"parts": [{"text": PERSONA}]},
        "contents": [{"parts": [{"text": text}]}]
    }
    try:
        r = requests.post(GEMINI_URL, json=payload, timeout=15)
        r.raise_for_status()
        return r.json()["candidates"][0]["content"]["parts"][0]["text"]
    except Exception as e:
        return f"Couldn't reach the assistant: {e}"

Enter your Gemini API key: ··········


## 8. Gradio Interface

Ties everything together: user uploads a CSV, sets income/goal, asks a
question, and gets both the budget breakdown and the assistant's reply
in one click.

In [32]:
def run_app(csv_file, income, goal, question):
    if csv_file is None:
        return "Please upload a CSV first.", ""

    summary = break_down_budget(csv_file.name, income, goal)
    if summary is None:
        return "Couldn't read that file. Check it has Amount and Category columns.", ""

    # Build the summary text using plain + concatenation for clarity
    summary_text = f"Total spent: ${summary['total_spent']:.2f}\n"
    summary_text = summary_text + f"Remaining: ${summary['remaining_balance']:.2f}\n"
    summary_text = summary_text + f"Goal achieved: {summary['goal_achieved']}\n\n"

    for cat, amt in summary["category_totals"].items():
        summary_text = summary_text + f"- {cat}: ${amt:.2f}\n"

    # Plain if-else (safe and easy to explain)
    if question:
        reply = ask_assistant(question, summary)
    else:
        reply = "Ask me something about your budget!"

    return summary_text, reply


# inputs = what the user provides, outputs = what the app shows back, fn = the function that connects them
demo = gr.Interface(
    fn=run_app,
    inputs=[
        gr.File(label="Transactions CSV"),
        gr.Number(label="Monthly income", value=1500),
        gr.Number(label="Savings goal", value=300),
        gr.Textbox(label="Ask the assistant")
    ],
    outputs=[
        gr.Textbox(label="Budget summary"),
        gr.Textbox(label="Assistant reply")
    ],
    title="Budget Breaker"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bf3ec7211902bc5a9d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
